# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. 
The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.
**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [5]:
pip install requests beautifulsoup4


   ---------------------------------------- 0/8 [urllib3]
   ---------------------------------------- 0/8 [urllib3]
   ---------------------------------------- 0/8 [urllib3]
   ---------------------------------------- 0/8 [urllib3]
   ---------------------------------------- 0/8 [urllib3]
   ---------- ----------------------------- 2/8 [soupsieve]
   --------------- ------------------------ 3/8 [idna]
   --------------- ------------------------ 3/8 [idna]
   -------------------- ------------------- 4/8 [charset_normalizer]
   -------------------- ------------------- 4/8 [charset_normalizer]
   -------------------- ------------------- 4/8 [charset_normalizer]
   -------------------- ------------------- 4/8 [charset_normalizer]
   ------------------------------ --------- 6/8 [requests]
   ------------------------------ --------- 6/8 [requests]
   ------------------------------ --------- 6/8 [requests]
   ----------------------------------- ---- 7/8 [beautifulsoup4]
   ------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from bs4 import BeautifulSoup
import pandas as pd
#upc urls don't work because it has ../../../ so need another librabry
from urllib.parse import urljoin
import requests


In [ ]:
def get_genres(base_url):
    response = requests.get(base_url)
    soup = BeautifulSoup(response.content, 'html.parser')
    genre_links = soup.find('ul', class_='nav nav-list').find('ul').find_all('a')
    
    genres = {genre.get_text(strip=True): urljoin(base_url, genre['href']) for genre in genre_links}
    return genres

def get_all_book_urls(base_url):
    page_urls = [base_url]
    book_urls = []
    
    for page_url in page_urls:
        response = requests.get(page_url)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Extract URLs for all books on the current page
        for h3 in soup.find_all('h3'):
            a_tag = h3.find('a')
            book_url = urljoin(base_url, a_tag['href'])
            book_urls.append(book_url)
        
        # Find the next page button and follow it
        next_button = soup.find('li', class_='next')
        if next_button:
            next_page = next_button.find('a')['href']
            new_page_url = urljoin(base_url, next_page)
            page_urls.append(new_page_url)

    return book_urls

def scrape_books(min_rating, max_price, desired_genre):
    all_books_data = []
    genres = get_genres(main_url)
    
    if desired_genre not in genres:
        print(f"Genre '{desired_genre}' not found.")
        return pd.DataFrame()

    book_urls = get_all_book_urls(genres[desired_genre])

    for book_url in book_urls:
        try:
            book_response = requests.get(book_url)
            book_response.raise_for_status()
            book_soup = BeautifulSoup(book_response.content, 'html.parser')

            # Extract book information
            upc = get_UPC(book_soup)
            title = get_title(book_soup)
            price_text = get_price(book_soup)
            price_value = float(price_text.lstrip('£'))
            rating = get_rating(book_soup)
            rating_value = convert_rating_to_number(rating)
            genre = get_book_genre(book_soup)
            availability = get_availability(book_soup)
            description = get_book_description(book_soup)

            # Display diagnostic info
            #print(f"Book: {title}, Price: {price_value}, Rating: {rating_value}, Genre: {genre}")

            # Apply filters based on rating, price, and desired genres
            if (rating_value >= min_rating and price_value <= max_price
                    and (genre == desired_genre)):
                book_data = {
                    "UPC": upc,
                    "Title": title,
                    "Price (£)": price_value,
                    "Rating": rating_value,
                    "Genre": genre,
                    "Availability": availability,
                    "Description": description,
                }
                all_books_data.append(book_data)
                #print(f"Added {title} to results.")

        except requests.exceptions.RequestException as e:
            print(f"Error fetching {book_url}: {e}")

    df = pd.DataFrame(all_books_data)
    if df.empty:
        print("Resulting DataFrame is empty.")
    return df

def convert_rating_to_number(rating_text):
    ratings = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    return ratings.get(rating_text, 0)

# Base URL for the main index page
main_url = "https://books.toscrape.com/catalogue/category/books_1/index.html"
desired_genre = 'Fantasy'  # Choose the desired genre
books_df = scrape_books(4.0, 20.0, desired_genre)
display(books_df)

,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,0e691eda369f4e09,Princess Between Worlds (Wide-Awake Princess #5),13.34,5,Fantasy,In stock (16 available),Just as Annie and Liam are busy making plans t...
1,4fa75fc6829431ca,Every Heart a Doorway (Every Heart A Doorway #1),12.16,5,Fantasy,In stock (7 available),Eleanor West’s Home for Wayward ChildrenNo Sol...
2,f2c8f4b4c94d5c5c,A Feast for Crows (A Song of Ice and Fire #4),17.21,4,Fantasy,In stock (5 available),"With A Feast for Crows, Martin delivers the lo..."
